Dependencies
terminal pip install commands

pip install pytorch-forecasting --extra-index-url https://download.pytorch.org/whl/cpu

In [116]:
import numpy as np
import pandas as pd
from pytorch_forecasting import TimeSeriesDataSet

In [128]:
class DataService:
    
    def __init__(self, formatted_data_sets):
        self.formatted_data_sets = formatted_data_sets
        
    def get_formatted_data_sets(self):
        return self.formatted_data_sets
        
    def format_data(self, data_frames):
        
        for index_value in data_frames.index: #data_frames.index[0:1:1]
            
            row = data_frames.loc[[index_value]]
            
            time_series_data = row.get(data_frames.columns[4:len(data_frames.columns)])
            
            time_series_data = time_series_data.set_axis(["Price"], axis=0)
            
            row.drop(columns=row.columns, inplace=True)
            
            row = row.assign(RegionID=[np.repeat(index_value, len(time_series_data.columns))])
            
            row = row.explode("RegionID")
            
            row.reset_index(drop=True, inplace=True)
            
            row = row.assign(Date=time_series_data.columns.T)
            
            row = row.assign(Price=time_series_data.T.to_numpy(copy=True))
            
            self.formatted_data_sets[index_value] = row

In [130]:
data_frames = pd.read_csv("Metro_new_con_median_sale_price_per_sqft_uc_sfrcondo_month.csv")
data_frames.set_index("RegionID", inplace=True)
data_frames.sort_index(inplace=True)

data_service = DataService({})
data_service.format_data(data_frames)
formatted_data_sets = data_service.get_formatted_data_sets()
#print(len(formatted_data_sets))
#print(formatted_data_sets[102001])

{102001:    RegionID        Date       Price
0    102001  2018-01-31  134.134666
1    102001  2018-02-28  134.826025
2    102001  2018-03-31  136.998984
3    102001  2018-04-30  136.775042
4    102001  2018-05-31  139.170897
..      ...         ...         ...
92   102001  2025-09-30  204.545455
93   102001  2025-10-31  206.698440
94   102001  2025-11-30  203.475870
95   102001  2025-12-31  201.694232
96   102001  2026-01-31  201.995012

[97 rows x 3 columns], 394299:    RegionID        Date       Price
0    394299  2018-01-31         NaN
1    394299  2018-02-28         NaN
2    394299  2018-03-31         NaN
3    394299  2018-04-30         NaN
4    394299  2018-05-31         NaN
..      ...         ...         ...
92   394299  2025-09-30  178.730159
93   394299  2025-10-31  186.346453
94   394299  2025-11-30  188.368293
95   394299  2025-12-31  186.869351
96   394299  2026-01-31  185.850338

[97 rows x 3 columns], 394308:    RegionID        Date       Price
0    394308  2018-01-31  18